# Word-Level Seq2Seq with Attention — English-Russian Translator

Train a GRU encoder-decoder with Bahdanau-style additive attention
(`nn.GRU` + `Attention`) to translate English sentences into Russian, on the
[opus-100](https://huggingface.co/datasets/Helsinki-NLP/opus-100) en-ru corpus.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F
from colorama import Fore, Style
from datasets import load_dataset
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

from dl_roadmap.data import Lang
from dl_roadmap.engine import TeacherForcedTrainer, TrainerConfig
from dl_roadmap.utils import LoggerConfig, save_model, seed_everything, setup_logger
from dl_roadmap.visualization import plot_training_history

In [ ]:
%matplotlib inline

pd.set_option("display.width", 150)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)

seed_everything()
setup_logger(LoggerConfig(disable_logging=True))

## Dataset

In [ ]:
dataset = load_dataset("Helsinki-NLP/opus-100", "en-ru")

train_df = pd.DataFrame(
    dataset["train"].shuffle(seed=42).select(range(200_000))["translation"]
)
val_df = pd.DataFrame(dataset["test"]["translation"])

### Overview

In [ ]:
print(f"{Fore.MAGENTA}DataFrame Info:{Style.RESET_ALL}")
for name, df in [("train", train_df), ("val", val_df)]:
    print(f"\n{Fore.CYAN}====== {name} ======{Style.RESET_ALL}")
    df.info()

In [ ]:
print(f"{Fore.YELLOW}First Rows of DataFrame:{Style.RESET_ALL}")
for name, df in [("train", train_df), ("val", val_df)]:
    print(f"\n{Fore.CYAN}====== {name} ======{Style.RESET_ALL}")
    print(df.head())

In [ ]:
print(f"{Fore.RED}Missing Values in Each Column:{Style.RESET_ALL}")
for name, df in [("train", train_df), ("val", val_df)]:
    print(f"\n{Fore.CYAN}====== {name} ======{Style.RESET_ALL}")
    missing = df.isnull().sum()
    missing_percent = (missing / len(df)) * 100
    missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_percent})
    print(missing_df)

### Preprocessing

In [ ]:
def prepare(df: pd.DataFrame) -> pd.DataFrame:
    """Lowercases the ``en``/``ru`` columns and splits them into word tokens.

    Args:
        df: DataFrame with raw ``en`` and ``ru`` text columns.

    Returns:
        The same DataFrame with ``en``/``ru`` replaced by lists of
        lowercase word/number tokens.
    """
    cols = ["en", "ru"]

    df[cols] = df[cols].apply(lambda col: col.str.lower())
    df[cols] = df[cols].apply(lambda col: col.str.findall(r"[а-яa-z0-9']+"))

    return df


train_df = prepare(train_df)
val_df = prepare(val_df)

In [ ]:
print(f"{Fore.YELLOW}First Rows of prepared DataFrame:{Style.RESET_ALL}")
for name, df in [("train", train_df), ("val", val_df)]:
    print(f"\n{Fore.CYAN}====== {name} ======{Style.RESET_ALL}")
    print(df.head())

### Length Filtering

In [ ]:
MAX_SEQ_LEN = 60


def filter_by_length(df: pd.DataFrame, max_len: int = MAX_SEQ_LEN) -> pd.DataFrame:
    """Drops sentence pairs where either side exceeds `max_len` tokens.

    Args:
        df: DataFrame with tokenized ``en``/``ru`` columns.
        max_len: Maximum number of tokens allowed per side (``<BOS>``/``<EOS>``
            excluded; the encoded sequence is ``max_len + 2`` tokens long).

    Returns:
        The filtered DataFrame, re-indexed.
    """
    mask = (df["en"].str.len() <= max_len) & (df["ru"].str.len() <= max_len)
    return df[mask].reset_index(drop=True)


print(f"{Fore.YELLOW}Rows before/after length filtering:{Style.RESET_ALL}\n")
for name, df in [("train", train_df), ("val", val_df)]:
    before = len(df)
    filtered = filter_by_length(df)
    print(f"{name}: {before} -> {len(filtered)}")

train_df = filter_by_length(train_df)
val_df = filter_by_length(val_df)

### Tokenization

In [ ]:
def build_lang(data: pd.Series, name: str) -> Lang:
    """Builds a vocabulary from a column of tokenized sentences.

    Args:
        data: Series of tokenized sentences (lists of words).
        name: Human-readable name of the language (e.g. ``"en"``).

    Returns:
        A ``Lang`` populated with every word seen in ``data``.
    """
    lang = Lang(name)

    for sentence in data:
        lang.add_sentence(sentence)

    return lang


input_lang = build_lang(train_df["en"], "en")
output_lang = build_lang(train_df["ru"], "ru")

In [ ]:
print(f"{Fore.GREEN}Vocabulary Sizes:{Style.RESET_ALL}\n")
for lang in (input_lang, output_lang):
    print(f"{lang.name}: {lang.n_words}")

### PyTorch Dataset

In [ ]:
class EnRuDataset(Dataset):
    """Sentence-pair dataset for English-Russian translation.

    Wraps a DataFrame of tokenized ``en``/``ru`` sentence pairs, encoding
    each side as a ``<BOS>``/``<EOS>``-wrapped sequence of vocabulary ids.
    """

    def __init__(self, df: pd.DataFrame, input_lang: Lang, output_lang: Lang) -> None:
        """Initializes the dataset.

        Args:
            df: DataFrame with tokenized ``en`` and ``ru`` columns.
            input_lang: Vocabulary used to encode the ``en`` column.
            output_lang: Vocabulary used to encode the ``ru`` column.
        """
        super().__init__()

        self.df = df
        self.input_lang = input_lang
        self.output_lang = output_lang

        self._input_unk_id = self.input_lang.get_by_word("<UNK>", -1)
        self._output_unk_id = self.output_lang.get_by_word("<UNK>", -1)

    def __len__(self) -> int:
        """Returns the number of sentence pairs in the dataset."""
        return len(self.df)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        """Returns the input/target token id sequences for a sentence pair.

        Args:
            idx: Index of the sentence pair in the dataset.

        Returns:
            A tuple of the source (``en``) and target (``ru``) token id
            sequences, both as long tensors.
        """
        row = self.df.iloc[idx]
        input_data, output_data = row["en"], row["ru"]

        x_raw = ["<BOS>", *list(input_data), "<EOS>"]
        y_raw = ["<BOS>", *list(output_data), "<EOS>"]

        x = torch.tensor(
            [self.input_lang.get_by_word(word, self._input_unk_id) for word in x_raw],
            dtype=torch.long,
        )
        y = torch.tensor(
            [self.output_lang.get_by_word(word, self._output_unk_id) for word in y_raw],
            dtype=torch.long,
        )

        return x, y

### Collation

In [ ]:
assert input_lang.get_by_word("<PAD>") == output_lang.get_by_word("<PAD>")

pad_id = input_lang.get_by_word("<PAD>")


def collate_fn(
    batch: list[tuple[torch.Tensor, torch.Tensor]],
) -> tuple[torch.Tensor, torch.Tensor]:
    """Pads a batch of source/target token id sequences to the same length.

    Args:
        batch: A list of (source, target) token id sequence pairs of
            variable length.

    Returns:
        A tuple of the padded source and target token id tensors, both
        padded with ``pad_id`` and shaped ``batch_size x max_seq_len``.
    """
    xs, ys = zip(*batch)

    xs = pad_sequence(xs, batch_first=True, padding_value=pad_id)
    ys = pad_sequence(ys, batch_first=True, padding_value=pad_id)

    return xs, ys

In [ ]:
train_dataset = EnRuDataset(train_df, input_lang, output_lang)
val_dataset = EnRuDataset(val_df, input_lang, output_lang)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
)

## Training

In [ ]:
class EncoderRNN(nn.Module):
    """GRU-based sequence encoder.

    Embeds token ids and runs them through a GRU, producing the
    per-step hidden states and the final hidden state used to initialize
    the decoder.
    """

    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int = 64,
        hidden_size: int = 128,
        num_layers: int = 1,
        dropout: float = 0.2,
        pad_id: int = 0,
    ) -> None:
        """Initializes the model.

        Args:
            vocab_size: Number of distinct tokens in the vocabulary.
            embedding_dim: Size of the token embedding vectors.
            hidden_size: Size of the RNN hidden state.
            num_layers: Number of stacked RNN layers.
            dropout: Dropout probability applied to the embedded input.
            pad_id: Token id used for padding, ignored by the embedding.
        """
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_id,
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Encodes a batch of input sequences.

        Args:
            x: Input token ids of shape ``batch_size x seq_len``.

        Returns:
            A tuple of the per-step hidden states, shaped
            ``batch_size x seq_len x hidden_size``, and the final hidden
            state, shaped ``num_layers x batch_size x hidden_size``.
        """
        embedding = self.dropout(self.embedding(x))
        output, hidden = self.gru(embedding)
        return output, hidden

In [ ]:
class Attention(nn.Module):
    """Bahdanau-style additive attention.

    Scores each encoder timestep against the current decoder hidden state
    with a small feed-forward network, then returns the weighted sum of
    encoder outputs (the context vector) alongside the attention weights.
    """

    def __init__(self, hidden_size: int) -> None:
        """Initializes the attention layers.

        Args:
            hidden_size: Size of the query and key vectors (the decoder's
                and encoder's hidden state size).
        """
        super().__init__()

        self.w_1 = nn.Linear(hidden_size, hidden_size)
        self.w_2 = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, 1)

    def forward(
        self, query: torch.Tensor, keys: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Computes a context vector for the current decoder step.

        Args:
            query: Current decoder hidden state, shaped
                ``batch_size x 1 x hidden_size``.
            keys: Encoder per-step hidden states, shaped
                ``batch_size x seq_len x hidden_size``.

        Returns:
            A tuple of the context vector, shaped
            ``batch_size x 1 x hidden_size``, and the attention weights
            over encoder timesteps, shaped ``batch_size x 1 x seq_len``.
        """
        scores = self.v(F.tanh(self.w_1(query) + self.w_2(keys)))
        scores = scores.squeeze(2).unsqueeze(1)

        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, keys)

        return context, weights

In [ ]:
class DecoderRNN(nn.Module):
    """GRU-based autoregressive decoder with attention.

    Starting from ``<BOS>``, decodes one token at a time for up to
    ``MAX_LENGTH`` steps, attending over the encoder outputs at each step.
    During training, if a target is given, each step is fed the
    ground-truth previous token (teacher forcing) for as long as the
    target sequence lasts; otherwise it greedily feeds back its own most
    likely output.
    """

    MAX_LENGTH = 64

    def __init__(
        self,
        hidden_size: int = 128,
        num_layers: int = 1,
        output_size: int = 64,
        bos_id: int = 1,
        dropout: float = 0.2,
    ) -> None:
        """Initializes the model.

        Args:
            hidden_size: Size of the RNN hidden state.
            num_layers: Number of stacked RNN layers.
            output_size: Number of distinct tokens in the vocabulary.
            bos_id: Token id used to seed the first decoding step.
            dropout: Dropout probability applied to the embedded input.
        """
        super().__init__()

        self.bos_id = bos_id

        self.dropout = nn.Dropout(dropout)
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = Attention(hidden_size)

        self.gru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

        self.fc = nn.Linear(hidden_size, output_size)

    def forward(
        self,
        encoder_outputs: torch.Tensor,
        encoder_hidden: torch.Tensor,
        target: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Decodes an output sequence from the encoder state.

        Args:
            encoder_outputs: Encoder per-step hidden states, attended over
                at each decoding step, shaped
                ``batch_size x seq_len x hidden_size``.
            encoder_hidden: Encoder final hidden state, used to initialize
                the decoder's hidden state.
            target: Ground-truth token ids of shape ``batch_size x seq_len``,
                used for teacher forcing while ``seq_len < MAX_LENGTH``. If
                None, decoding is free-running (greedy) for every step.

        Returns:
            A tuple of the log-probabilities over the vocabulary for each
            decoded step, shaped ``batch_size x MAX_LENGTH x output_size``,
            the decoder's final hidden state, and the attention weights for
            each decoded step, shaped ``batch_size x MAX_LENGTH x seq_len``.
        """
        device = encoder_outputs.device
        target_length = target.size(1) if target is not None else 0

        batch_size = encoder_outputs.size(0)
        decoder_input = torch.full(
            (batch_size, 1),
            self.bos_id,
            dtype=torch.long,
            device=device,
        )
        hidden = encoder_hidden
        outputs = []
        attentions = []

        for index in range(self.MAX_LENGTH):
            output, hidden, attn_weights = self.forward_step(
                dec_input=decoder_input,
                hidden=hidden,
                enc_outputs=encoder_outputs,
            )
            outputs.append(output)
            attentions.append(attn_weights)

            if index < target_length:
                # Teacher forcing: feed the ground-truth token.
                decoder_input = target[:, index].unsqueeze(1)
            else:
                # Greedy decoding: feed the most likely token back in.
                _, topi = output.topk(1)
                decoder_input = topi.squeeze(-1).detach()

        outputs = torch.cat(outputs, dim=1)
        outputs = F.log_softmax(outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return outputs, hidden, attentions

    def forward_step(
        self,
        dec_input: torch.Tensor,
        hidden: torch.Tensor,
        enc_outputs: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Runs a single decoding step.

        Args:
            dec_input: Token ids for the current step, shaped
                ``batch_size x 1``.
            hidden: Decoder hidden state from the previous step.
            enc_outputs: Encoder per-step hidden states, attended over to
                produce this step's context vector, shaped
                ``batch_size x seq_len x hidden_size``.

        Returns:
            A tuple of the output logits for the current step, shaped
            ``batch_size x 1 x output_size``, the updated hidden state, and
            the attention weights over encoder timesteps, shaped
            ``batch_size x 1 x seq_len``.
        """
        embedding = self.dropout(self.embedding(dec_input))

        query = hidden.permute(1, 0, 2)
        context, attn_weights = self.attention(query, enc_outputs)
        dec_input = torch.cat((embedding, context), dim=2)

        output, hidden = self.gru(dec_input, hidden)
        output = self.fc(output)

        return output, hidden, attn_weights

In [ ]:
class Translator(nn.Module):
    """Attention-based encoder-decoder model for English-Russian translation."""

    def __init__(self, input_lang: Lang, output_lang: Lang) -> None:
        """Initializes the encoder and decoder.

        Args:
            input_lang: Source-language (``en``) vocabulary.
            output_lang: Target-language (``ru``) vocabulary, whose size
                sets the decoder's output size and whose ``<BOS>`` id seeds
                decoding.
        """
        super().__init__()

        bos_id = output_lang.get_by_word("<BOS>")

        self.encoder = EncoderRNN(len(input_lang))
        self.decoder = DecoderRNN(output_size=len(output_lang), bos_id=bos_id)

    def forward(self, x: torch.Tensor, y: torch.Tensor | None = None) -> torch.Tensor:
        """Encodes ``x`` and decodes its translation.

        Args:
            x: Input token ids of shape ``batch_size x seq_len``.
            y: Target token ids of shape ``batch_size x seq_len``, used for
                teacher forcing. If None, decoding is free-running: each
                step's predicted token is fed back in as the next input.

        Returns:
            Log-probabilities over the vocabulary for each decoded step,
            shaped ``batch_size x MAX_LENGTH x vocab_size``.
        """
        enc_output, enc_hidden = self.encoder.forward(x)
        dec_output, _, _ = self.decoder.forward(enc_output, enc_hidden, y)

        return dec_output

In [ ]:
model = Translator(input_lang, output_lang)

ce = nn.NLLLoss(ignore_index=pad_id)


def loss_fn(preds: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """Computes NLL loss over flattened sequence predictions.

    Args:
        preds: Log-probabilities of shape ``batch_size x seq_len x vocab_size``.
        targets: Target token ids of shape ``batch_size x seq_len``.

    Returns:
        The scalar NLL loss, ignoring ``pad_id`` targets.
    """
    preds = preds[:, : targets.size(1), :]
    return ce(preds.reshape(-1, preds.size(-1)), targets.reshape(-1))


opt = torch.optim.Adam(model.parameters(), lr=1e-3)

trainer_config = TrainerConfig(
    epochs=45,
    checkpoint_dir="../checkpoints",
    checkpoint_every=1,
    patience=2,
    min_delta=1e-2,
    restore_best_weights=True,
)
trainer = TeacherForcedTrainer(model, opt, loss_fn, config=trainer_config)
trainer.fit(train_loader, val_loader)

save_model(model, "../models/translator")

## Results

In [ ]:
plot_training_history(**trainer.history, best_epoch=trainer.best_epoch)